# GCP Full Pipeline - Bronze to Silver to Gold

**The complete data engineering pipeline: ingest, clean, model, query.**

## Architecture

**Architecture:**
```
Source files --> GCS (Bronze) --> BigQuery raw --> BigQuery silver --> BigQuery gold --> Reports
```
[See full diagram on GitHub](https://github.com/sunilmogadati/systems-in-production/blob/main/systems/production-diagnostics/architecture.md)

## Where Data Lives at Each Step

| Step | Storage | What's There | Tables |
|---|---|---|---|
| **Source** | Your laptop / GitHub | Raw files (JSON, CSV) | -- |
| **Bronze** | GCS bucket + BigQuery `raw` dataset | Exact copy of source files | raw.calls, raw.orders, raw.campaigns, raw.products |
| **Silver** | BigQuery `silver` dataset | Cleaned tables (same shape, better quality) | silver.calls, silver.orders |
| **Gold** | BigQuery `gold` dataset | Star schema (new structure for business) | dim_date, dim_time, dim_campaign, dim_disposition, fact_calls |

Bronze and Silver have tables named after the source (calls, orders). **Gold restructures into facts and dimensions** - a new concept introduced only at the Gold layer.

| Step | What It Does | BigQuery Dataset |
|---|---|---|
| **Step 0** | Setup: auth, billing, bucket, APIs | -- |
| **Step 1: Bronze** | Upload raw files to GCS, load into BigQuery | `raw` |
| **Step 1a** | Incremental Bronze loading (production pattern) | `raw` |
| **Step 2: Silver** | Dedup, timezone, types, flag nulls | `silver` |
| **Step 2a** | Incremental Silver loading (MERGE pattern) | `silver` |
| **Step 3: Gold** | Star schema: dimensions + fact table | `gold` |
| **Step 4: Quality Gates** | Validate: no dupes, no orphans, counts match | `gold` |
| **Step 5: Reports** | Business queries on the star schema | `gold` |


In [ ]:
# === CONFIGURATION ===
PROJECT_ID = "data-pipeline-project-490820"
BUCKET_NAME = f"de-accelerator-{PROJECT_ID}"
LOCATION = "us-central1"

# Three BigQuery datasets — one per layer
RAW_DATASET = "raw"           # Bronze: raw tables loaded from GCS
SILVER_DATASET = "silver"     # Silver: cleaned, deduped, timezone fixed
GOLD_DATASET = "gold"         # Gold: star schema (dimensions + facts)

print(f"Project:  {PROJECT_ID}")
print(f"Bucket:   gs://{BUCKET_NAME}/")
print(f"Datasets: {RAW_DATASET} → {SILVER_DATASET} → {GOLD_DATASET}")

In [ ]:
# === AUTHENTICATE ===
import os
if 'COLAB_RELEASE_TAG' in os.environ:
    from google.colab import auth
    auth.authenticate_user()
    print("Authenticated via Colab.")
else:
    print("Running locally. Use 'gcloud auth login' if not authenticated.")

!gcloud config set project {PROJECT_ID}

---

## How GCP Permissions Work (IAM)

**IAM** (Identity and Access Management, pronounced "I-am") controls who can do what in your GCP project.

### Why Your Commands Work Right Now

When you ran `gcloud auth login`, you authenticated with your personal Google account. If you created the project, you're the **Owner**. Owners can do anything: create buckets, load data, run queries, delete resources.

**IAM Permissions:**
```
You (human, Owner role) --> can do everything
Cloud Function (service account) --> needs BigQuery Data Editor + Storage Object Viewer
Dataproc (service account) --> needs Storage Object Admin + BigQuery Data Editor
```
[See full diagram on GitHub](https://github.com/sunilmogadati/systems-in-production/blob/main/foundations/data/cloud-pipeline/02_Concepts.md)

This is fine for learning. It is NOT how production works.

### How Production Works

In a real company, you would NOT be the Owner. You would have specific roles:

| Role | What It Allows | Who Gets It |
|---|---|---|
| **Owner** | Everything (dangerous) | Project creator, CTO |
| **Editor** | Create and modify resources (can't change IAM) | Senior engineers |
| **Viewer** | Read-only (can't change anything) | Analysts, auditors |
| **BigQuery Data Editor** | Load data, create tables, run queries | Data engineers |
| **Storage Object Viewer** | Read files from GCS | Pipeline jobs, analysts |
| **Storage Object Admin** | Read + write + delete files in GCS | Pipeline jobs |

### The Key Concept: Service Accounts

When YOU run commands, your Google account is used. When a **Cloud Function** or **Dataproc job** runs automatically, it uses a **service account** (a robot identity with its own permissions).

**Manual vs Automated:**
```
Manual (this notebook): You type every command
Automated (production): File lands -> Function triggers -> Pipeline runs automatically
```
[See full diagram on GitHub](https://github.com/sunilmogadati/systems-in-production/blob/main/foundations/data/cloud-pipeline/04_Automation.md)

Service accounts start with minimal permissions. You must explicitly grant what they need. This is the **principle of least privilege**: give each identity only the permissions it needs, nothing more.

### What This Means for You

In this notebook, everything works because you're the Owner. In the [GCP Pipeline Automation](GCP_Pipeline_Automation.ipynb) notebook, we set up IAM for service accounts because automation runs without your personal account.

**Interview question:** "What is the principle of least privilege?" Answer: Every user and service should have only the minimum permissions needed to do their job. If a pipeline only needs to read from GCS and write to BigQuery, it should not have permission to delete buckets or modify IAM policies.


In [ ]:
# === IAM: See your current permissions ===

print("Your current IAM roles on this project:")
print("=" * 50)

import subprocess, json as j

# Get your authenticated email
result = subprocess.run(["gcloud", "auth", "list", "--format=json"], capture_output=True, text=True)
accounts = j.loads(result.stdout)
active = [a for a in accounts if a.get('status') == 'ACTIVE']
if active:
    email = active[0]['account']
    print(f"Authenticated as: {email}")
else:
    print("No active account found. Run: gcloud auth login")
    email = None

if email:
    result = subprocess.run(
        ["gcloud", "projects", "get-iam-policy", PROJECT_ID, "--format=json"],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        policy = j.loads(result.stdout)
        my_roles = [b['role'].split('/')[-1] for b in policy.get('bindings', [])
                     if f"user:{email}" in b.get('members', [])]
        if my_roles:
            for role in my_roles:
                print(f"  - {role}")
        else:
            print("  (No explicit roles found - may have inherited access)")

print("\nIf you see 'owner' or 'editor', that is why all commands work.")
print("In production, you would have specific roles like 'bigquery.dataEditor'.")


#### Same Step in GCP Console: View IAM Permissions

1. Go to [console.cloud.google.com/iam-admin](https://console.cloud.google.com/iam-admin)
2. You see a table of all members and their roles
3. Find your email address in the list
4. **You Should See:** "Owner" or "Editor" next to your name
5. Also notice the service accounts (ending in `@developer.gserviceaccount.com`)

**Key things to notice:**
- Your personal account has Owner/Editor (that's why everything works)
- Service accounts exist but may have limited roles
- In the [Automation notebook](GCP_Pipeline_Automation.ipynb), we grant additional roles to service accounts so automated pipeline steps work


In [ ]:
# === HELPER FUNCTION ===
# Runs BigQuery SQL and prints results.
# WHY: Colab's ! shell commands don't expand Python variables in single-quoted SQL.

import subprocess

def run_bq(sql, print_results=True):
    """Run a BigQuery SQL query. Returns stdout."""
    result = subprocess.run(["bq", "query", "--use_legacy_sql=false", "--format=prettyjson", sql],
                            capture_output=True, text=True)
    if print_results:
        print(result.stdout)
    if result.returncode != 0:
        print("ERROR:", result.stderr)
    return result.stdout

# Fully qualified table prefix
R = f"{PROJECT_ID}.{RAW_DATASET}"       # raw tables
S = f"{PROJECT_ID}.{SILVER_DATASET}"    # silver tables
G = f"{PROJECT_ID}.{GOLD_DATASET}"      # gold tables

print(f"Raw:    {R}.calls")
print(f"Silver: {S}.calls")
print(f"Gold:   {G}.fact_calls")

---

## Step 1: BRONZE — Upload to GCS and Load into BigQuery

Bronze = raw data, untouched. Exact copy of what the source system produced.

In [ ]:
# Get the data files (Colab: clone from GitHub)
import os
if not os.path.exists("../data") and not os.path.exists("data"):
    print("Downloading data from GitHub...")
    !git clone --depth 1 https://github.com/sunilmogadati/systems-in-production.git /tmp/de-repo 2>/dev/null
    DATA_DIR = "/tmp/de-repo/data"
elif os.path.exists("../data/calls.json"):
    DATA_DIR = "../data"
else:
    DATA_DIR = "data"

print(f"Data: {DATA_DIR}")
print(f"Files: {os.listdir(DATA_DIR)}")

In [ ]:
# Create bucket (skip if exists)
!gcloud storage buckets create gs://{BUCKET_NAME}/ --location={LOCATION} 2>&1 || echo "Bucket exists."

# Upload to Bronze layer in GCS
!gcloud storage cp {DATA_DIR}/calls.json gs://{BUCKET_NAME}/bronze/
!gcloud storage cp {DATA_DIR}/campaigns.csv gs://{BUCKET_NAME}/bronze/
!gcloud storage cp {DATA_DIR}/orders.csv gs://{BUCKET_NAME}/bronze/
!gcloud storage cp {DATA_DIR}/products.csv gs://{BUCKET_NAME}/bronze/

print("\nBronze layer in GCS:")
!gcloud storage ls gs://{BUCKET_NAME}/bronze/

#### Same Step in GCP Console: Create a Storage Bucket

1. Go to [console.cloud.google.com/storage](https://console.cloud.google.com/storage)
2. Click **"Create"** (blue button, top)
3. **Name:** Enter your bucket name (must be globally unique)
4. **Location type:** Region
5. **Region:** us-central1 (Iowa)
6. **Storage class:** Standard
7. **Access control:** Uniform (default)
8. Click **"Create"**

**You Should See:** Your new bucket in the storage browser, empty, with 0 objects.

**Upload files to the bucket:**
1. Click your bucket name to open it
2. Click **"Create folder"** and name it `bronze`
3. Open the `bronze` folder
4. Click **"Upload files"** and select calls.json, orders.csv, campaigns.csv, products.csv
5. **You Should See:** 4 files listed in the bronze/ folder


In [ ]:
# Create RAW dataset in BigQuery
!bq mk --dataset --location={LOCATION} {PROJECT_ID}:{RAW_DATASET} 2>&1 || echo "Dataset exists."

# Load raw files into BigQuery
!bq load --autodetect --replace --source_format=NEWLINE_DELIMITED_JSON {RAW_DATASET}.calls gs://{BUCKET_NAME}/bronze/calls.json
!bq load --autodetect --replace --source_format=CSV {RAW_DATASET}.campaigns gs://{BUCKET_NAME}/bronze/campaigns.csv
!bq load --autodetect --replace --source_format=CSV {RAW_DATASET}.orders gs://{BUCKET_NAME}/bronze/orders.csv
!bq load --autodetect --replace --source_format=CSV {RAW_DATASET}.products gs://{BUCKET_NAME}/bronze/products.csv

print("\nRaw tables:")
!bq ls {RAW_DATASET}

#### Same Step in GCP Console: Create BigQuery Datasets and Load Data

**Create datasets:**
1. Go to [console.cloud.google.com/bigquery](https://console.cloud.google.com/bigquery)
2. In the left panel, click the **three dots** next to your project name
3. Click **"Create dataset"**
4. **Dataset ID:** `raw`
5. **Data location:** us-central1
6. Click **"Create dataset"**
7. Repeat for `silver` and `gold`

**You Should See:** Three datasets (raw, silver, gold) under your project in the left panel.

**Load data from GCS into BigQuery:**
1. Click the `raw` dataset in the left panel
2. Click **"Create table"** (top right)
3. **Source:** Google Cloud Storage
4. **GCS path:** Browse to your bucket > bronze > calls.json
5. **File format:** JSON (newline delimited)
6. **Table name:** calls
7. Check **"Auto detect"** for schema
8. Click **"Create table"**
9. Repeat for campaigns.csv, orders.csv, products.csv (change format to CSV)

**You Should See:** Four tables (calls, campaigns, orders, products) under the raw dataset.

**Query in BigQuery Console:**
1. Click **"Compose a new query"** (top)
2. Type: `SELECT COUNT(*) FROM raw.calls`
3. Click **"Run"**
4. **You Should See:** Row count at the bottom (510 rows)


In [ ]:
# Quick look at the raw data — what did we load?
run_bq(f"SELECT COUNT(*) AS total_calls FROM `{R}.calls`")
run_bq(f"SELECT COUNT(*) AS total_campaigns FROM `{R}.campaigns`")
run_bq(f"SELECT COUNT(*) AS total_orders FROM `{R}.orders`")

---

### Step 1a: Incremental Bronze Loading (Production Pattern)

What we did above is a **full load**: upload all files, load all data. This works for demos and initial loads. In production, new data arrives daily (or hourly). You only load what's new.

**Full load vs Incremental load:**

| Approach | What Happens | When to Use |
|---|---|---|
| **Full load** | Re-upload and re-load everything | Initial setup, small data, recovery from failure |
| **Incremental load** | Only upload and load new files since last run | Daily production runs |

**How incremental Bronze works:**

*(Diagram: see GitHub for visual version)*
[View on GitHub](https://github.com/sunilmogadati/systems-in-production/blob/main/foundations/data/cloud-pipeline/02_Concepts.md)

**Key differences from full load:**

```bash
# FULL LOAD (what we did above) — replaces everything
bq load --replace --source_format=CSV raw.campaigns gs://bucket/bronze/campaigns.csv

# INCREMENTAL LOAD — appends new data, keeps existing
bq load --append --source_format=NEWLINE_DELIMITED_JSON \
    raw.calls \
    gs://bucket/bronze/calls/calls_2026_03_15.json
```

**The `--replace` vs `--append` flag is the difference.** Replace wipes the table. Append adds to it.

**What you need for incremental Bronze:**
- **File naming convention**: Include dates in filenames (`calls_2026_03_15.json`) so you know what's new
- **Landing zone tracking**: Record which files have been loaded (a simple tracking table or Cloud Function that fires on file arrival)
- **Dedup in Silver**: If the same file gets loaded twice by accident, Silver's dedup step catches it

**GCS notification pattern (automated):**

**Incremental Load Flow:**
```
New raw data arrives --> Identify new rows (WHERE load_date > last_run) --> Transform only new rows --> MERGE into silver table
```
[See full diagram on GitHub](https://github.com/sunilmogadati/systems-in-production/blob/main/foundations/data/cloud-pipeline/04_Automation.md)

This is how production pipelines work: a file arrives, a notification fires, a function loads it, and a record is kept. No human runs commands manually.

For this notebook, we use full load (simple, repeatable). The incremental pattern is what you build when you move to production.


---

## Step 2: SILVER - Clean, Dedup, Unify

### How Silver Works

**Silver Transform Flow:**
```
BigQuery raw.calls (510 rows, has duplicates, UTC timestamps)
    |
    v  SQL: CREATE TABLE silver.* AS SELECT (dedup, clean, timezone fix)
    |
BigQuery silver.calls (~500 rows, unique, local timestamps, nulls flagged)
```
[See full diagram on GitHub](https://github.com/sunilmogadati/systems-in-production/blob/main/foundations/data/cloud-pipeline/02_Concepts.md)

**Where does Silver live?** In BigQuery, in a dataset called `silver`. Not in a GCS bucket. The SQL `CREATE OR REPLACE TABLE` reads from `raw.*` and writes to `silver.*` in one step.

**Why not a separate bucket?** For our data size, BigQuery handles both storage and compute. At terabyte scale, you would use Spark on Dataproc to transform and write Parquet files to GCS, then load into BigQuery. Same logic, different execution engine.

Silver is the **single source of truth** for all downstream consumers.

| Silver DOES | Silver Does NOT |
|---|---|
| Remove duplicates | Drop records with missing values |
| Convert timezones (UTC to local) | Join to dimension tables |
| Fix data types | Create surrogate keys |
| Standardize text (casing, whitespace) | Apply business rules |
| Flag missing values (boolean columns) | Build the star schema |


In [ ]:
# Create Silver dataset
!bq mk --dataset --location={LOCATION} {PROJECT_ID}:{SILVER_DATASET} 2>&1 || echo "Dataset exists."

#### Same Step in GCP Console: Create Silver Dataset

The `silver` dataset was created above. Now we create tables via SQL.

1. In BigQuery Console, click **"Compose a new query"**
2. Paste the SQL from the code cell above (the `CREATE OR REPLACE TABLE silver.calls AS ...` query)
3. Click **"Run"**
4. **You Should See:** "This statement created a new table silver.calls" in the results panel
5. Repeat for silver.orders

**Verify in the left panel:** Expand the `silver` dataset. You should see `calls` and `orders` tables.

**Click on silver.calls > Preview tab:** You should see clean data with no duplicate call_ids, local timezone timestamps, and standardized text.


In [ ]:
# === SILVER: calls ===
# Dedup by call_id, convert timezone, fix types, flag nulls, standardize text

run_bq(f"""
CREATE OR REPLACE TABLE `{S}.calls` AS

WITH cleaned AS (
    SELECT
        call_id,
        CAST(dnis AS STRING) AS dnis,
        caller_phone AS caller_ani,

        -- Timezone: convert UTC → Eastern ONCE, here in Silver
        -- Every downstream query uses these local times — no more AT TIME ZONE anywhere else
        DATETIME(start_time, 'US/Eastern') AS call_started_local,
        DATETIME(end_time, 'US/Eastern') AS call_ended_local,
        DATE(DATETIME(start_time, 'US/Eastern')) AS call_date_local,
        EXTRACT(HOUR FROM DATETIME(start_time, 'US/Eastern')) AS call_hour_local,

        duration AS duration_sec,
        LOWER(TRIM(disposition)) AS disposition,
        UPPER(TRIM(channel)) AS call_type,

        -- Flag nulls (do NOT drop — that is a Gold-layer decision)
        duration IS NULL AS has_missing_duration,
        disposition IS NULL AS has_missing_disposition,

        -- Dedup: if same call_id appears twice, keep the first by start_time
        ROW_NUMBER() OVER (PARTITION BY call_id ORDER BY start_time) AS row_num

    FROM `{R}.calls`
)

SELECT * EXCEPT(row_num)
FROM cleaned
WHERE row_num = 1
""")

print("silver.calls created.")

In [ ]:
# === SILVER: orders ===
# Dedup by order_id, fix types

run_bq(f"""
CREATE OR REPLACE TABLE `{S}.orders` AS

SELECT * EXCEPT(row_num)
FROM (
    SELECT *,
        ROW_NUMBER() OVER (PARTITION BY order_id ORDER BY order_date DESC) AS row_num
    FROM `{R}.orders`
)
WHERE row_num = 1
""")

print("silver.orders created.")

#### Same Step in GCP Console: Silver Transforms

The Silver tables are created by running SQL in BigQuery. You can do this in the console:

1. Go to [console.cloud.google.com/bigquery](https://console.cloud.google.com/bigquery)
2. Click **"Compose a new query"**
3. Paste the `CREATE OR REPLACE TABLE silver.calls AS SELECT ...` query from the cell above
4. Click **"Run"**
5. **You Should See:** "This statement created a new table silver.calls" in the results panel
6. Repeat for silver.orders with its query
7. Expand the `silver` dataset in the left panel to verify both tables exist
8. Click **silver.calls > Preview** tab to see the clean data (no duplicates, local timestamps, standardized text)


In [ ]:
# === VERIFY SILVER ===
# How many records survived cleaning? Any missing values?

print("=== Silver Verification ===")
run_bq(f"""
SELECT
    'raw.calls' AS source, COUNT(*) AS rows FROM `{R}.calls`
UNION ALL
SELECT
    'silver.calls', COUNT(*) FROM `{S}.calls`
UNION ALL
SELECT
    'raw.orders', COUNT(*) FROM `{R}.orders`
UNION ALL
SELECT
    'silver.orders', COUNT(*) FROM `{S}.orders`
""")

print("Missing values in silver.calls:")
run_bq(f"""
SELECT
    COUNT(*) AS total,
    COUNTIF(has_missing_duration) AS missing_duration,
    COUNTIF(has_missing_disposition) AS missing_disposition,
    COUNT(DISTINCT call_id) AS unique_call_ids
FROM `{S}.calls`
""")

#### Same Step in GCP Console: Verify Silver

1. In BigQuery Console, run the verification query above
2. **You Should See:** A table comparing raw vs silver row counts
3. Silver count should be less than raw (duplicates removed)
4. Check for missing values: the query shows how many nulls exist in key columns


---

### Step 2a: Incremental Loading (Production Pattern)

Everything above uses **full reload**: drop the table, recreate from scratch. That works for demos and small data. In production, you need **incremental loading**: only process NEW data since the last run.

**Why incremental matters:**

| Approach | 1,000 rows | 1,000,000 rows | 1,000,000,000 rows |
|---|---|---|---|
| Full reload | 2 seconds | 5 minutes | Hours, expensive |
| Incremental | 2 seconds | 2 seconds | 2 seconds |

Full reload reprocesses everything every time. Incremental only processes what changed. At scale, the difference is hours vs seconds and dollars vs cents.

**How incremental works in BigQuery:**

*(Diagram: see GitHub for visual version)*
[View on GitHub](https://github.com/sunilmogadati/systems-in-production/blob/main/foundations/data/cloud-pipeline/02_Concepts.md)

**The key SQL pattern: MERGE**

```sql
-- Instead of CREATE OR REPLACE (full reload):
MERGE INTO silver.calls AS target
USING (
    -- Only new rows since last run
    SELECT * FROM raw.calls
    WHERE _PARTITIONTIME > TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 1 DAY)
) AS source
ON target.call_id = source.call_id
WHEN NOT MATCHED THEN INSERT (...)
WHEN MATCHED THEN UPDATE SET ...
```

**What you need for incremental:**
- A way to identify "new" rows (timestamp column, partition, or watermark)
- MERGE statement instead of CREATE OR REPLACE
- A tracking table or watermark that records "last successful run"
- Error handling: what if the same data arrives twice?

This applies to Silver AND Gold. The full reload pattern we use here is step 1. Incremental is step 2 when you move to production.


---

## Step 3: GOLD - Star Schema (Facts and Dimensions)

This is where the data model changes.

**Bronze and Silver have tables named after the source:** `calls`, `orders`, `campaigns`, `products`. Same shape as the source. Just cleaner.

**Gold restructures the data for business questions.** It introduces two new concepts:

| Concept | What It Is | Example | Analogy |
|---|---|---|---|
| **Dimension** | Context/description table. WHO, WHAT, WHEN, WHERE. Changes slowly. | `dim_campaign` (campaign name, client, channel) | The labels on a filing cabinet |
| **Fact** | Event/measurement table. WHAT HAPPENED. One row per event. Has numbers. | `fact_calls` (call_id, duration, keys to dimensions) | The documents inside the cabinet |

**Star Schema (Entity Relationship):**
```
fact_calls --> dim_date (call_date_key)
fact_calls --> dim_time (call_hour_key)
fact_calls --> dim_campaign (campaign_key)
fact_calls --> dim_disposition (disposition_key)
```
[See full diagram on GitHub](https://github.com/sunilmogadati/systems-in-production/blob/main/systems/production-diagnostics/architecture.md)

**Why not just query Silver directly?**

| Query on Silver | Query on Gold (Star Schema) |
|---|---|
| You figure out the joins every time | Pre-defined relationships via keys |
| You calculate metrics every time | Pre-computed in the fact table |
| Different analysts write different joins | Everyone uses the same model |
| Slow on large data (many joins at query time) | Fast (pre-joined, pre-aggregated) |

Gold is where the data becomes **self-service**. An analyst can query it without knowing how the source systems work.

### Gold Dimensions: Build Context Tables

Dimensions provide meaning to the raw numbers. They map codes to names, dates to day-of-week, DNIS to campaigns.

| Dimension | What It Answers | Changes When |
|---|---|---|
| dim_date | What day, month, year, day-of-week? | Never (calendar is fixed) |
| dim_time | What hour of day? Peak vs off-peak? | Never |
| dim_campaign | Which campaign, client, channel? | New campaign launches |
| dim_disposition | How did the call end? (completed, abandoned, etc.) | New disposition types added |


In [ ]:
# Create Gold dataset
!bq mk --dataset --location={LOCATION} {PROJECT_ID}:{GOLD_DATASET} 2>&1 || echo "Dataset exists."

#### Same Step in GCP Console: Create Gold Tables

Same process as Silver. The Gold tables are also created by SQL.

1. In BigQuery Console, click **"Compose a new query"**
2. Paste the dimension SQL from the code cells above (dim_date, dim_time, dim_campaign, dim_disposition)
3. Run each one
4. **You Should See:** Four dimension tables in the `gold` dataset

**Verify the star schema:**
1. Expand the `gold` dataset in the left panel
2. You should see: dim_date, dim_time, dim_campaign, dim_disposition
3. Click each one > **Schema tab** to see the columns
4. Click **Preview** to see sample data

After running the fact_calls SQL:
5. You should also see `fact_calls` in the gold dataset
6. Click fact_calls > Schema: should have surrogate keys (date_key, campaign_key, etc.) pointing to the dimensions


In [ ]:
# === dim_date ===
# Pre-computed calendar. Timezone already converted.
# Every query joins on date_key (integer) and gets day_name, is_weekend, month — no parsing.

run_bq(f"""
CREATE OR REPLACE TABLE `{G}.dim_date` AS
SELECT
    CAST(FORMAT_DATE('%Y%m%d', d) AS INT64) AS date_key,
    d AS full_date,
    FORMAT_DATE('%A', d) AS day_name,
    EXTRACT(MONTH FROM d) AS month_num,
    FORMAT_DATE('%B', d) AS month_name,
    EXTRACT(QUARTER FROM d) AS quarter,
    EXTRACT(YEAR FROM d) AS year,
    CASE WHEN EXTRACT(DAYOFWEEK FROM d) IN (1, 7) THEN TRUE ELSE FALSE END AS is_weekend,
    DATE_TRUNC(d, ISOWEEK) AS monday_week
FROM UNNEST(GENERATE_DATE_ARRAY('2026-01-01', '2026-12-31')) AS d
""")

print("dim_date: 365 rows")

In [ ]:
# === dim_time ===
# 24 rows — one per hour. Replaces 96 half-hour columns in flat table approaches.

run_bq(f"""
CREATE OR REPLACE TABLE `{G}.dim_time` AS
SELECT
    hour AS time_key,
    hour,
    CONCAT(LPAD(CAST(hour AS STRING), 2, '0'), ':00') AS hour_label,
    CASE
        WHEN hour BETWEEN 6 AND 11 THEN 'Morning'
        WHEN hour BETWEEN 12 AND 17 THEN 'Afternoon'
        WHEN hour BETWEEN 18 AND 21 THEN 'Evening'
        ELSE 'Night'
    END AS period
FROM UNNEST(GENERATE_ARRAY(0, 23)) AS hour
""")

print("dim_time: 24 rows")

In [ ]:
# === dim_campaign ===
# Maps DNIS (phone number) → client, campaign, channel.
# Business logic is DATA here, not code.

run_bq(f"""
CREATE OR REPLACE TABLE `{G}.dim_campaign` AS
SELECT
    ROW_NUMBER() OVER (ORDER BY dnis) AS campaign_key,
    CAST(dnis AS STRING) AS dnis,
    campaign_name,
    client_name,
    channel,
    start_date,
    end_date
FROM `{R}.campaigns`
""")

run_bq(f"SELECT * FROM `{G}.dim_campaign`")

In [ ]:
# === dim_disposition ===
# Maps disposition text → key + boolean flags.
# Replaces CASE WHEN logic that would otherwise be in every query.

run_bq(f"""
CREATE OR REPLACE TABLE `{G}.dim_disposition` AS
SELECT
    ROW_NUMBER() OVER (ORDER BY disposition) AS disposition_key,
    disposition AS disposition_name,
    CASE WHEN disposition = 'completed' THEN TRUE ELSE FALSE END AS is_sale,
    CASE WHEN disposition IN ('voicemail', 'abandoned') THEN TRUE ELSE FALSE END AS is_abandon,
    CASE WHEN disposition = 'callback' THEN TRUE ELSE FALSE END AS is_callback
FROM (
    SELECT DISTINCT disposition
    FROM `{S}.calls`
    WHERE disposition IS NOT NULL
)
""")

run_bq(f"SELECT * FROM `{G}.dim_disposition`")

#### Same Step in GCP Console: Gold Dimensions

1. In BigQuery Console, run each dimension SQL one at a time (dim_date, dim_time, dim_campaign, dim_disposition)
2. After each: expand the `gold` dataset in the left panel, verify the new table appears
3. Click each dimension > **Schema** tab to see the columns
4. Click **Preview** to verify data looks right
5. **You Should See:** dim_date, dim_time, dim_campaign, dim_disposition all listed under the gold dataset


---

## Step 4: GOLD FACTS — The Join

The fact table is where Silver (clean records) meets Gold Dimensions (business context).

| Silver | Gold Dimensions | Gold Fact Table |
|:---|:---|:---|
| Clean records, one per call | Context: campaign, date, disposition | **Both combined: call record + integer dimension keys** |
| String fields (`dnis = '8005551001'`) | Surrogate keys (`campaign_key = 7`) | Integer key joins — fast, consistent |
| Rebuilt nightly | Maintained as reference data | Rebuilt nightly from Silver + Dimensions |

In [ ]:
# === fact_calls ===
# One row per call. Surrogate dimension keys. Partitioned by date. Clustered by campaign.

run_bq(f"""
CREATE OR REPLACE TABLE `{G}.fact_calls`
PARTITION BY call_date
CLUSTER BY campaign_key
AS
SELECT
    ROW_NUMBER() OVER (ORDER BY s.call_id) AS call_key,

    -- Dimension keys (integer — fast joins)
    CAST(FORMAT_DATE('%Y%m%d', s.call_date_local) AS INT64) AS date_key,
    s.call_hour_local AS time_key,
    dc.campaign_key,
    dd.disposition_key,

    -- Source reference
    s.call_id,
    s.call_type,
    s.call_date_local AS call_date,

    -- Measures
    s.duration_sec,
    o.total AS order_total,
    o.order_id,
    CASE WHEN o.order_id IS NOT NULL THEN TRUE ELSE FALSE END AS is_order,

    -- Caller info
    s.dnis,
    s.caller_ani,

    -- Quality flags (carried from Silver)
    s.has_missing_duration,
    s.has_missing_disposition

FROM `{S}.calls` s
LEFT JOIN `{G}.dim_campaign` dc ON s.dnis = dc.dnis
LEFT JOIN `{G}.dim_disposition` dd ON s.disposition = dd.disposition_name
LEFT JOIN `{S}.orders` o ON s.call_id = o.call_id
""")

print("fact_calls created.")

#### Same Step in GCP Console: Fact Table

1. Run the fact_calls SQL in BigQuery Console
2. **You Should See:** `fact_calls` appears in the gold dataset
3. Click **Schema** tab: should show surrogate keys (date_key, campaign_key, etc.) alongside call data
4. Click **Preview**: each row is one call with keys pointing to dimension tables
5. Quick verification query:
```sql
SELECT f.call_id, d.campaign_name, f.duration_sec
FROM gold.fact_calls f
JOIN gold.dim_campaign d ON f.campaign_key = d.campaign_key
LIMIT 5
```
This confirms the star schema joins work.


---

## Step 5: VERIFY — Data Quality Gates

Before anyone queries Gold, verify the pipeline produced correct results. If any gate fails, stop and investigate.

In [ ]:
# Gate 1: No duplicate call_ids in fact table
print("Gate 1: Duplicate check")
run_bq(f"""
SELECT call_id, COUNT(*) AS dupes
FROM `{G}.fact_calls`
GROUP BY call_id HAVING COUNT(*) > 1
""")
# Expected: empty result (no duplicates)

#### Same Step in GCP Console: Run Quality Gates

1. In BigQuery Console, paste each quality gate query and run
2. **Gate 1 (duplicates):** Should return 0 rows (no duplicates)
3. **Gate 2 (orphan keys):** Should return 0 rows (all keys match)
4. **Gate 3 (row count):** Silver and Gold counts should be close
5. **Gate 4 (orphan orders):** Shows orders without matching calls (expected - these are real data quality issues)

**If any gate fails:** The query returns rows. That means there's a data quality issue to investigate.


In [ ]:
# Gate 2: Orphan dimension keys (calls without campaign or disposition match)
print("Gate 2: Orphan keys")
run_bq(f"""
SELECT
    COUNTIF(campaign_key IS NULL) AS missing_campaign,
    COUNTIF(disposition_key IS NULL) AS missing_disposition,
    COUNT(*) AS total
FROM `{G}.fact_calls`
""")
# Investigate any non-zero values — means a DNIS or disposition isn't in the dimension table

In [ ]:
# Gate 3: Row count — Silver vs Gold should match (or be close)
print("Gate 3: Row count comparison")
run_bq(f"""
SELECT 'silver.calls' AS layer, COUNT(*) AS rows FROM `{S}.calls`
UNION ALL
SELECT 'gold.fact_calls', COUNT(*) FROM `{G}.fact_calls`
""")
# Should be the same count. If Gold has fewer, some calls didn't join to any dimension.

In [ ]:
# Gate 4: Orphan orders (orders without a matching call)
print("Gate 4: Orphan orders")
run_bq(f"""
SELECT COUNT(*) AS orphan_orders
FROM `{S}.orders` o
LEFT JOIN `{G}.fact_calls` f ON o.call_id = f.call_id
WHERE f.call_key IS NULL
""")
# Non-zero = orders exist for calls that are not in the fact table. Investigate.

#### Same Step in GCP Console: Quality Gates 2-4

Run each gate query in BigQuery Console:
- **Gate 2** (orphan keys): Should return 0 rows. If not, a call references a campaign or disposition that doesn't exist in the dimension tables.
- **Gate 3** (row count): Silver and Gold counts should be close. A large gap means the fact table join is dropping rows.
- **Gate 4** (orphan orders): May return rows. This is a real data quality finding: orders exist without a matching call. Investigate, don't ignore.

If any gate returns unexpected rows, investigate before using Gold data for reports or ML.


---

## When to Use Spark/Dataproc (Scaling Beyond BigQuery SQL)

Everything above runs as BigQuery SQL. That works well for:
- Data up to ~100GB
- Transforms expressible in SQL
- Teams comfortable with SQL

When you need Spark/Dataproc:

| Scenario | Why BigQuery SQL Isn't Enough |
|---|---|
| **Data > 100GB per load** | Spark distributes processing across many machines |
| **Complex transforms** | Python/PySpark gives more flexibility than SQL (ML feature engineering, custom parsing) |
| **Multiple output formats** | Write Parquet, Delta Lake, Iceberg (not just BigQuery tables) |
| **Cost optimization** | Process in Spark (cheap compute), load results into BigQuery (expensive storage+query) |

### What Are Parquet, Delta Lake, and Iceberg?

These are file formats for storing data. Each adds more capability:

| Format | What It Is | Analogy |
|---|---|---|
| **Parquet** | A file format optimized for analytics. Stores data in columns, not rows. Fast to read, compact. | A filing cabinet where all the "names" are in one drawer, all the "dates" in another. Finding all names is instant. |
| **Delta Lake** | Parquet files + a transaction log. Adds ACID transactions, time travel (query yesterday's data), MERGE/UPDATE/DELETE on files. Created by Databricks. | Same filing cabinet, but now with a logbook tracking every change. You can undo mistakes and see what the cabinet looked like yesterday. |
| **Iceberg** | Similar to Delta Lake but open standard (not tied to one vendor). Used by Netflix, Apple, AWS. | Same concept as Delta Lake, different manufacturer. Like choosing between two brands of the same tool. |

**Why this matters:** Plain Parquet files can get corrupted if two jobs write at the same time. Delta Lake and Iceberg prevent that with transaction logs. For this notebook, we use BigQuery tables (simplest). When you need file-based storage with transactions, Delta Lake or Iceberg is the next step.

### How Dataproc Fits in the Pipeline

**Small Data vs Large Data Path:**
```
Small data (what we did):   GCS Bronze --> bq load --> BigQuery raw --> SQL --> silver --> SQL --> gold
Large data (production):    GCS Bronze --> Dataproc PySpark --> GCS Silver (Parquet) --> bq load --> BigQuery --> SQL --> gold
```
[See full diagram on GitHub](https://github.com/sunilmogadati/systems-in-production/blob/main/foundations/data/cloud-pipeline/06_Scale.md)

### How Spark Data Gets to BigQuery Reports

Spark writes Parquet files to GCS, not directly to BigQuery. To query this data in BigQuery (for Gold reports), you need one extra step: `bq load`.

```
Spark job writes     -->  GCS: gs://bucket/silver/calls/*.parquet
bq load command      -->  BigQuery: silver.calls (now queryable with SQL)
Gold SQL runs        -->  BigQuery: gold.fact_calls (star schema)
Reports query        -->  BigQuery: gold.* (dashboards, ML)
```

In this notebook, we skip Spark entirely and do everything in BigQuery SQL. The Spark path is shown for when you need to scale beyond what SQL can handle. In the [GCP Pipeline Automation](GCP_Pipeline_Automation.ipynb) notebook, we run the full Spark path: Dataproc writes Parquet to GCS, then `bq load` brings it into BigQuery for Gold queries.

### What a Spark Silver Transform Looks Like

```python
# silver_transform.py
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, row_number, lower, trim
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("SilverTransform").getOrCreate()

# Read from Bronze in GCS
calls = spark.read.json("gs://my-bucket/bronze/calls.json")

# Same logic as our BigQuery SQL, but in PySpark
window = Window.partitionBy("call_id").orderBy("start_time")
silver_calls = (calls
    .withColumn("row_num", row_number().over(window))
    .filter(col("row_num") == 1)
    .drop("row_num")
    .withColumn("disposition", lower(trim(col("disposition"))))
)

# Write to Silver in GCS as Parquet (partitioned by date)
silver_calls.write.mode("overwrite").partitionBy("call_date").parquet(
    "gs://my-bucket/silver/calls/"
)
```

Then submit to Dataproc:
```bash
gcloud dataproc jobs submit pyspark \
    gs://my-bucket/code/silver_transform.py \
    --cluster=my-pipeline-cluster \
    --region=us-central1
```

**The thinking is identical.** Dedup, clean, standardize. The only difference is the execution engine (Spark instead of BigQuery SQL) and the output location (GCS Parquet instead of BigQuery table). An extra `bq load` step brings the Parquet data into BigQuery for Gold queries and reports.


---

## Step 6: QUERY — Reports on the Star Schema

The payoff. Simple joins on integer keys. No timezone logic. No dedup. No tribal knowledge.

These are the same questions from the Pandas Analytics Pipeline — now running on the star schema.

In [ ]:
# Report 1: Campaign Performance — conversion rate, revenue, duration

print("=== Campaign Performance ===")
run_bq(f"""
SELECT
    dc.client_name,
    dc.campaign_name,
    dc.channel,
    COUNT(*) AS total_calls,
    COUNTIF(f.is_order) AS orders,
    ROUND(COUNTIF(f.is_order) / COUNT(*) * 100, 1) AS conversion_pct,
    ROUND(SUM(f.order_total), 2) AS total_revenue,
    ROUND(AVG(f.duration_sec), 1) AS avg_duration
FROM `{G}.fact_calls` f
JOIN `{G}.dim_campaign` dc ON f.campaign_key = dc.campaign_key
GROUP BY dc.client_name, dc.campaign_name, dc.channel
ORDER BY total_revenue DESC
""")

#### Same Step in GCP Console: Run Reports

1. In BigQuery Console, paste each report query and run
2. Each report shows results in a table at the bottom
3. Click the **"Save results"** button to export as CSV
4. Click **"Explore data"** > **"Explore with Looker Studio"** to create a chart directly from the query result

**Looker Studio shortcut:**
1. After running any report query, click **"Explore with Looker Studio"**
2. Looker Studio opens with your data pre-loaded
3. Choose a chart type (bar, line, pie)
4. The dashboard is live and connected to your BigQuery data

This is how analysts actually work: write a query in BigQuery, then click through to a dashboard.


In [ ]:
# Report 2: Hourly Volume — when do we need more agents?

print("=== Hourly Volume ===")
run_bq(f"""
SELECT
    dt.hour,
    dt.period,
    COUNT(*) AS total_calls,
    COUNTIF(f.is_order) AS orders,
    ROUND(AVG(f.duration_sec), 1) AS avg_duration
FROM `{G}.fact_calls` f
JOIN `{G}.dim_time` dt ON f.time_key = dt.time_key
GROUP BY dt.hour, dt.period
ORDER BY dt.hour
""")

In [ ]:
# Report 3: Day-of-Week Pattern

print("=== Day of Week ===")
run_bq(f"""
SELECT
    dd.day_name,
    dd.is_weekend,
    COUNT(*) AS total_calls,
    COUNTIF(f.is_order) AS orders,
    ROUND(COUNTIF(f.is_order) / COUNT(*) * 100, 1) AS conversion_pct
FROM `{G}.fact_calls` f
JOIN `{G}.dim_date` dd ON f.date_key = dd.date_key
GROUP BY dd.day_name, dd.is_weekend
ORDER BY total_calls DESC
""")

In [ ]:
# Report 4: Disposition Breakdown — why are calls ending the way they do?

print("=== Disposition Breakdown ===")
run_bq(f"""
SELECT
    di.disposition_name,
    di.is_sale,
    di.is_abandon,
    COUNT(*) AS call_count,
    ROUND(COUNT(*) / SUM(COUNT(*)) OVER () * 100, 1) AS pct_of_total
FROM `{G}.fact_calls` f
JOIN `{G}.dim_disposition` di ON f.disposition_key = di.disposition_key
GROUP BY di.disposition_name, di.is_sale, di.is_abandon
ORDER BY call_count DESC
""")

In [ ]:
# Report 5: The VP's Question — conversion by campaign, by day, by time period

print("=== Conversion by Campaign x Day x Time Period ===")
run_bq(f"""
SELECT
    dc.campaign_name,
    dc.channel,
    dd.day_name,
    dt.period,
    COUNT(*) AS total_calls,
    COUNTIF(f.is_order) AS orders,
    ROUND(COUNTIF(f.is_order) / COUNT(*) * 100, 1) AS conversion_pct
FROM `{G}.fact_calls` f
JOIN `{G}.dim_campaign` dc ON f.campaign_key = dc.campaign_key
JOIN `{G}.dim_date` dd ON f.date_key = dd.date_key
JOIN `{G}.dim_time` dt ON f.time_key = dt.time_key
GROUP BY dc.campaign_name, dc.channel, dd.day_name, dt.period
ORDER BY dc.campaign_name, conversion_pct DESC
""")

In [ ]:
# Report 6: Data Quality Summary

print("=== Data Quality Summary ===")
run_bq(f"""
SELECT
    COUNT(*) AS total_calls,
    COUNTIF(campaign_key IS NULL) AS no_campaign_match,
    COUNTIF(disposition_key IS NULL) AS no_disposition_match,
    COUNTIF(has_missing_duration) AS missing_duration,
    COUNTIF(has_missing_disposition) AS missing_disposition,
    COUNTIF(is_order) AS total_orders,
    ROUND(SUM(order_total), 2) AS total_revenue
FROM `{G}.fact_calls`
""")

#### Same Step in GCP Console: All Reports

Run each report query (1-6) in BigQuery Console. For any report:
1. Click **"Save results"** to export as CSV or Google Sheets
2. Click **"Explore data"** > **"Explore with Looker Studio"** for instant charts
3. Looker Studio opens with your data pre-loaded. Choose chart type (bar, line, pie).
4. The dashboard is live, connected to BigQuery.

**Tip:** Click **"Save query"** in BigQuery to save your favorite reports. They appear under "Saved queries" in the left panel for quick access.


---

## Summary — What Was Built

| Layer | Dataset | Tables | Purpose |
|:---|:---|:---|:---|
| **Bronze** | GCS bucket + `raw` | calls, campaigns, orders, products | Raw data, untouched |
| **Silver** | `silver` | calls, orders | Cleaned: deduped, timezone fixed, types correct, nulls flagged |
| **Gold Dimensions** | `gold` | dim_date, dim_time, dim_campaign, dim_disposition | Business context as data |
| **Gold Facts** | `gold` | fact_calls | One row per call, dimension keys, measures |

```
Source Systems → GCS Bronze → BigQuery Raw → Silver (clean) → Gold Dimensions + Gold Facts → Reports
```

Each layer has one job. A bug in one layer does not cascade to the others.

In [ ]:
# Final count summary
print("=== Pipeline Complete ===")
for dataset, table in [(R, 'calls'), (R, 'campaigns'), (R, 'orders'),
                        (S, 'calls'), (S, 'orders'),
                        (G, 'dim_date'), (G, 'dim_time'), (G, 'dim_campaign'),
                        (G, 'dim_disposition'), (G, 'fact_calls')]:
    result = subprocess.run(
        ["bq", "query", "--use_legacy_sql=false", "--format=csv",
         f"SELECT COUNT(*) as c FROM `{dataset}.{table}`"],
        capture_output=True, text=True
    )
    count = result.stdout.strip().split('\n')[-1] if result.returncode == 0 else 'ERROR'
    print(f"  {dataset}.{table}: {count} rows")

In [ ]:
# === CLEANUP (uncomment to delete everything) ===
# !bq rm -r -f {RAW_DATASET}
# !bq rm -r -f {SILVER_DATASET}
# !bq rm -r -f {GOLD_DATASET}
# !gcloud storage rm -r gs://{BUCKET_NAME}/
# print("All resources deleted.")

#### Same Step in GCP Console: Cleanup

To delete resources via console:
- **GCS bucket:** Go to [Storage](https://console.cloud.google.com/storage) > check your bucket > click **"Delete"**
- **BigQuery datasets:** Go to [BigQuery](https://console.cloud.google.com/bigquery) > click the three dots next to each dataset (raw, silver, gold) > **"Delete dataset"**
- **Or keep them:** The data is small enough to stay within the free tier. No cost if you leave it.
